<div style="text-align: center;">
    <h1 style="color: #FF6347;">Self-Guided Lab: Retrieval-Augmented Generation (RAGs)</h1>
</div>

<div style="text-align: center;">
    <img src="https://media4.giphy.com/media/v1.Y2lkPTc5MGI3NjExZ3FsdzRveTBrenMxM3VnbDMwaTJxN2NnZm50aGFibXk1NzNnY2Q0MCZlcD12MV9pbnRlcm5hbF9naWZfYnlfaWQmY3Q9Zw/LR5ZBwZHv02lmpVoEU/giphy.gif" alt="NLP Gif" style="width: 300px; height: 150px; object-fit: cover; object-position: center;">
</div>

<h1 style="color: #FF6347;">Data Storage & Retrieval</h1>


<h2 style="color: #FF8C00;">PyPDFLoader</h2>

`PyPDFLoader` is a lightweight Python library designed to streamline the process of loading and parsing PDF documents for text processing tasks. It is particularly useful in Retrieval-Augmented Generation workflows where text extraction from PDFs is required.

- **What Does PyPDFLoader Do?**
  - Extracts text from PDF files, retaining formatting and layout.
  - Simplifies the preprocessing of document-based datasets.
  - Supports efficient and scalable loading of large PDF collections.

- **Key Features:**
  - Compatible with popular NLP libraries and frameworks.
  - Handles multi-page PDFs and embedded images (e.g., OCR-compatible setups).
  - Provides flexible configurations for structured text extraction.

- **Use Cases:**
  - Preparing PDF documents for retrieval-based systems in RAGs.
  - Automating the text extraction pipeline for document analysis.
  - Creating datasets from academic papers, technical manuals, and reports.


In [1]:
%pip install langchain langchain_community pypdf
%pip install termcolor langchain_openai langchain-huggingface sentence-transformers chromadb langchain_chroma tiktoken openai python-dotenv


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [6]:
import os
# from langchain.docs_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader
#from langchain.text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
import warnings
warnings.filterwarnings('ignore')


<h3 style="color: #FF8C00;">Loading the Documents</h3>

In [7]:
# File path for the document

file_path = "../ai-for-everyone.pdf"

<h3 style="color: #FF8C00;">Documents into pages</h3>

The `PyPDFLoader` library allows efficient loading and splitting of PDF documents into smaller, manageable parts for NLP tasks.

This functionality is particularly useful in workflows requiring granular text processing, such as Retrieval-Augmented Generation (RAG).


In [8]:
# Load and split the document
loader = PyPDFLoader(file_path)
pages = loader.load_and_split()
len(pages)

297

<h3 style="color: #FF8C00;">Pages into Chunks</h3>


####  RecursiveCharacterTextSplitter in LangChain

The `RecursiveCharacterTextSplitter` is the **recommended splitter** in LangChain when you want to break down long documents into smaller, semantically meaningful chunks — especially useful in **RAG pipelines**, where clean context chunks lead to better LLM responses.

####  Parameters

| Parameter       | Description                                                                 |
|-----------------|-----------------------------------------------------------------------------|
| `chunk_size`    | The **maximum number of characters** allowed in a chunk (e.g., `1000`).     |
| `chunk_overlap` | The number of **overlapping characters** between consecutive chunks (e.g., `200`). This helps preserve context continuity. |

####  How it works
`RecursiveCharacterTextSplitter` attempts to split the text **intelligently**, trying the following separators in order:
1. Paragraphs (`"\n\n"`)
2. Lines (`"\n"`)
3. Sentences or words (`" "`)
4. Individual characters (as a last resort)

This makes it ideal for handling **natural language documents**, such as PDFs, articles, or long reports, without breaking sentences or paragraphs in awkward ways.



In [9]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(pages)

len(chunks)

1096

####  Alternative: CharacterTextSplitter

`CharacterTextSplitter` is a simpler splitter that breaks text into chunks based **purely on character count**, without trying to preserve any natural language structure.

##### Example:
```python
from langchain.text_splitter import CharacterTextSplitter

text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
````

This method is faster and more predictable but may split text in the middle of a sentence or paragraph, which can hurt performance in downstream tasks like retrieval or QA.

---

#### Comparison Table

| Feature                        | RecursiveCharacterTextSplitter | CharacterTextSplitter     |
| ------------------------------ | ------------------------------ | ------------------------- |
| Structure-aware splitting      |  Yes                          |  No                      |
| Preserves sentence/paragraphs  |  Yes                          |  No                      |
| Risk of splitting mid-sentence |  Minimal                     |  High                   |
| Ideal for RAG/document QA      |  Highly recommended           |  Only if structured text |
| Performance speed              |  Slightly slower             |  Faster                  |

---

#### Recommendation

Use `RecursiveCharacterTextSplitter` for most real-world document processing tasks, especially when building RAG pipelines or working with structured natural language content like PDFs or articles.

## Best Practices for Choosing Chunk Size in RAG

### Best Practices for Chunk Size in RAG

| Factor                      | Recommendation                                                                                                                                                                                          |
| ---------------------------| ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **LLM context limit**       | Choose a chunk size that lets you retrieve multiple chunks **without exceeding the model’s token limit**. For example, GPT-4o supports 128k tokens, but with GPT-3.5 (16k) or GPT-4 (32k), keep it modest. |
| **Chunk size (in characters)** | Typically: **500–1,000 characters** per chunk → ~75–200 tokens. This fits well for retrieval + prompt without context overflow.                                                                           |
| **Chunk size (in tokens)**  | If using token-based splitter (e.g. `TokenTextSplitter`): aim for **100–300 tokens** per chunk.                                                                                                            |
| **Chunk overlap**           | Use **overlap of 10–30%** (e.g., 100–300 characters or ~50 tokens) to preserve context across chunk boundaries and avoid cutting off important ideas mid-sentence.                                        |
| **Document structure**      | Use **`RecursiveCharacterTextSplitter`** to preserve semantic boundaries (paragraphs, sentences) instead of arbitrary cuts.                                                                                |
| **Task type**               | For **question answering**, smaller chunks (~500–800 chars) reduce noise.<br>For **summarization**, slightly larger chunks (~1000–1500) are OK.                                                          |
| **Embedding model**         | Some models (e.g., `text-embedding-3-large`) can handle long input. But still, smaller chunks give **finer-grained retrieval**, which improves relevance.                                                  |
| **Query type**              | If users ask **very specific questions**, small focused chunks are better. For broader queries, bigger chunks might help.                                                                                  |


### Rule of Thumb

| Use Case                 | Chunk Size      | Overlap |
| ------------------------| --------------- | ------- |
| Factual Q&A              | 500–800 chars   | 100–200 |
| Summarization            | 1000–1500 chars | 200–300 |
| Technical documents      | 400–700 chars   | 100–200 |
| Long reports/books       | 800–1200 chars  | 200–300 |
| Small LLMs (≤16k tokens) | ≤800 chars      | 100–200 |


### Avoid

- Chunks >2000 characters: risks context overflow.
- No overlap: may lose key information between chunks.



<h2 style="color: #FF8C00;">Embeddings</h2>

Embeddings transform text into dense vector representations, capturing semantic meaning and contextual relationships. They are essential for efficient document retrieval and similarity analysis.

- **What are OpenAI Embeddings?**
  - Pre-trained embeddings like `text-embedding-3-large` generate high-quality vector representations for text.
  - Encapsulate semantic relationships in the text, enabling robust NLP applications.

- **Key Features of `text-embedding-3-large`:**
  - Large-scale embedding model optimized for accuracy and versatility.
  - Handles diverse NLP tasks, including retrieval, classification, and clustering.
  - Ideal for applications with high-performance requirements.

- **Benefits:**
  - Reduces the need for extensive custom training.
  - Provides state-of-the-art performance in retrieval-augmented systems.
  - Compatible with RAGs to create powerful context-aware models.


In [11]:
# from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from dotenv import load_dotenv

In [12]:
load_dotenv()

True

In [13]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

<h2 style="color: #FF8C00;">ChromaDB</h2>

ChromaDB is a versatile vector database designed for efficiently storing and retrieving embeddings. It integrates seamlessly with embedding models to enable high-performance similarity search and context-based retrieval.

### Workflow Overview:
- **Step 1:** Generate embeddings using a pre-trained model (e.g., OpenAI's `text-embedding-3-large`).
- **Step 2:** Store the embeddings in ChromaDB for efficient retrieval and similarity calculations.
- **Step 3:** Use the stored embeddings to perform searches, matching, or context-based retrieval.

### Key Features of ChromaDB:
- **Scalability:** Handles large-scale datasets with optimized indexing and search capabilities.
- **Speed:** Provides fast and accurate retrieval of embeddings for real-time applications.
- **Integration:** Supports integration with popular frameworks and libraries for embedding generation.

In [15]:
# from langchain.vectorstores import Chroma
from langchain_chroma import Chroma

In [16]:
db = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db_LAB")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


<h1 style="color: #FF6347;">Retrieving Documents</h1>


### Exercise1: Write a user question that someone might ask about your book’s topic or content.

In [17]:
user_question = "What are three specific applications of AI in healthcare?" # User question
retrieved_docs = db.similarity_search(user_question, k=10) # k is the number of documents to retrieve

In [18]:
# Display top results
for i, doc in enumerate(retrieved_docs[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content[36:1000]}") # Display content

Document 1:
n AI applications will be able to plan, solve and reason 
problems independently, even for actions for which they have not been pro -
grammed initially. Such artificial general intelligence (AGI) will thus be able 
to broaden its horizons autonomously, entering new areas and domains. For 
example, an AGI-powered system could, on top of conveying news headlines 
during one’s morning routine, also learn to make coffee for the aforementioned 
individual preparing for work.
Finally, we might potentially even experience artificial super intelligence 
(ASI), the third generation of AI. Such truly self-conscious and self-aware 
AI systems, outperforming humans in (nearly) all domains, capable of gen -
eral wisdom, scientific creativity and social skills, could render human beings 
redundant. As such, in our above example, the individual would not need to 
prepare for work anymore, as this could be done entirely by the ASI-powered
Document 2:
al impact on their work and jobs (Shook

<h2 style="color: #FF8C00;">Preparing Content for GenAI</h2>

In [19]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [20]:
# Generate a formatted context from the retrieved documents
formatted_context = _get_document_prompt(retrieved_docs)
print("Context formatted for GPT model.")

Context formatted for GPT model.


<h2 style="color: #FF8C00;">ChatBot Architecture</h2>

### Exercice2: Write a prompt that is relevant and tailored to the content and style of your book.

In [21]:
# Extract page numbers from retrieved documents
page_numbers = []
for doc in retrieved_docs:
    page_num = doc.metadata.get('page', 'N/A')  # Get page number from metadata
    if page_num not in page_numbers:  # Avoid duplicates
        page_numbers.append(page_num)

# Convert to string (e.g., "15, 23, 31")
source_pages = ", ".join([str(p) for p in page_numbers])

print(f"   Pages: {source_pages}")

   Pages: 34, 38, 31, 98, 132, 32, 162, 89, 160, 35


In [22]:
prompt = f"""
You are a teaching assistant for the "AI for Everyone" book, helping beginners understand AI concepts.

Context from the book:
{formatted_context}

Source Page(s): {source_pages}

User Question: {user_question}

Instructions:
- Begin your answer by stating: "Based on information retrieved from the 'AI for Everyone' book:"
- Answer the question using ONLY the information provided in the context above
- Do not add information beyond what is stated in the context
- If the context does not contain enough information to fully answer the question, state what you CAN answer and clearly indicate what information is missing
- If the context does not contain the answer at all, respond with: "The answer to this question is outside the scope of the provided source. Please use alternative resources."
- Cite page numbers from the source when providing information (e.g., "Page 15")
- Use a clear, educational, and beginner-friendly tone
- Structure your response using markdown formatting with bullet points
- For each main point, use this nested structure:
  * **Title**: Brief statement
    - Definition/Explanation: Detailed explanation of the concept
    - Example: A practical, real-world example
- Provide a concise but complete answer (typically 2-5 bullet points, adjust based on complexity)

Answer:

"""


In [23]:
import openai

### Exercice3: Tune parameters like temperature, and penalties to control how creative, focused, or varied the model's responses are.

In [24]:
# Set up GPT client and parameters
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.4,  # Increase creativity
    'max_tokens': 800,  # Allow for longer responses
    'top_p': 0.9,        # Use nucleus sampling
    'frequency_penalty': 0.3,  # Reduce repetition
    'presence_penalty':  0.1  # Encourage new topics
}

<h1 style="color: #FF6347;">Response</h1>


In [25]:
messages = [{'role': 'user', 'content': prompt}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)

In [26]:
answer = completion.choices[0].message.content
print(answer)

Based on information retrieved from the 'AI for Everyone' book:

* **Identifying Infected Individuals**: 
  - **Definition/Explanation**: AI has been utilized to efficiently identify potentially infected humans during health crises, such as the COVID-19 pandemic.
  - **Example**: Researchers employed AI algorithms to analyze data and identify individuals who may have contracted the virus (Page 38).

* **Analyzing Viruses**: 
  - **Definition/Explanation**: AI can be used to analyze viruses, helping researchers understand their behavior and characteristics.
  - **Example**: AJ Venkatakrishnan applied AI to discover that a mutation of the original COVID-19 virus mimicked a protein that regulates fluid and salt equilibrium in the human body (Page 38).

* **Testing Treatments and Therapies**: 
  - **Definition/Explanation**: AI aids in testing possible treatments and therapies for diseases, improving the efficiency of medical research.
  - **Example**: During the pandemic, AI was leveraged

<img src="https://miro.medium.com/v2/resize:fit:824/1*GK56xmDIWtNQAD_jnBIt2g.png" alt="NLP Gif" style="width: 500px">

<h2 style="color: #FF6347;">Cosine Similarity</h2>

**Cosine similarity** is a metric used to measure the alignment or similarity between two vectors, calculated as the cosine of the angle between them. It is the **most common metric used in RAG pipelines** for vector retrieval.. It provides a scale from -1 to 1:

- **-1**: Vectors are completely opposite.
- **0**: Vectors are orthogonal (uncorrelated or unrelated).
- **1**: Vectors are identical.


<img src="https://storage.googleapis.com/lds-media/images/cosine-similarity-vectors.original.jpg" alt="NLP Gif" style="width: 700px">

<h2 style="color: #FF6347;">Keyword Highlighting</h2>

Highlighting important keywords helps users quickly understand the relevance of the retrieved text to their query.

In [27]:
from termcolor import colored

The `highlight_keywords` function is designed to highlight specific keywords within a given text. It replaces each keyword in the text with a highlighted version using the `colored` function from the `termcolor` library.


In [28]:
def highlight_keywords(text, keywords):
    for keyword in keywords:
        text = text.replace(keyword, colored(keyword, 'green', attrs=['bold']))
    return text

### Exercice4: add your keywords

In [29]:
query_keywords = [
    # Question terms
    'AI', 'artificial intelligence',
    'healthcare', 'medical', 'health',
    'application', 'applications',
    
    # Look for specific examples
    'diagnosis', 'treatment', 'imaging', 'surgery',
    'drug', 'patient', 'disease', 'detection'
] # add your keywords
for i, doc in enumerate(retrieved_docs[:1]):
    snippet = doc.page_content[:200]
    highlighted = highlight_keywords(snippet, query_keywords)
    print(f"Snippet {i+1}:\n{highlighted}\n{'-'*80}")

Snippet 1:
26 AI for Everyone?
Second-generation AI applications will be able to plan, solve and reason 
problems independently, even for actions for which they have not been pro -
grammed initially. Such artifi
--------------------------------------------------------------------------------


1. `query_keywords` is a list of keywords to be highlighted.
2. The loop iterates over the first document in retrieved_docs.
3. For each document, a snippet of the first 200 characters is extracted.
4. The highlight_keywords function is called to highlight the keywords in the snippet.
5. The highlighted snippet is printed along with a separator line.

<h1 style="color: #FF6347;">Bonus</h1>

**Try loading one of your own PDF books and go through the steps again to explore how the pipeline works with your content**:


## Part 1: Importing the libraries

In [30]:
# Standard library imports
import os
import warnings
warnings.filterwarnings('ignore')

# Third-party imports
from dotenv import load_dotenv
import openai

# LangChain imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# Design, visual effect 
from termcolor import colored  # For highlighting keywords
from IPython.display import Markdown, display

# Load environment variables
load_dotenv()


True

## Part 2: Loading and splitting the document

In [31]:
# File path for the document

file_path2 = "../WuXi_Apptec_IND_Guide.pdf"

In [32]:
# Load and split the document to pages
loader_bonus = PyPDFLoader(file_path2)
pages_bonus = loader_bonus.load_and_split()
len(pages_bonus)

25

In [33]:
# Split the pages to chunks
text_splitter2 = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks_bonus = text_splitter2.split_documents(pages_bonus)

len(chunks_bonus)

57

In [34]:
# Validation of chunks' output
print(f"✅ Created {len(chunks_bonus)} chunks")
print(f"\n📊 Chunk Statistics:")
print(f"   - Average chunk size: {sum(len(c.page_content) for c in chunks_bonus) / len(chunks_bonus):.0f} characters")
print(f"   - Configured overlap: 200 characters (20%)")

# Look at a sample chunk
print(f"\n📄 Sample Chunk (first one):")
print(f"   - Length: {len(chunks_bonus[0].page_content)} characters")
print(f"   - Page: {chunks_bonus[0].metadata.get('page', 'N/A')}")
print(f"   - Preview:\n")
print(chunks_bonus[0].page_content[:300] + "...")


✅ Created 57 chunks

📊 Chunk Statistics:
   - Average chunk size: 811 characters
   - Configured overlap: 200 characters (20%)

📄 Sample Chunk (first one):
   - Length: 159 characters
   - Page: 0
   - Preview:

THE ULTIMATE GUIDE TO PRECLINICAL IND STUDIES
IND, 
Enabled.
A thorough review of preclinical testing & submission requirements for 
Investigational New Drugs....


In [35]:
# Looking at the distribution:
import matplotlib.pyplot as plt

chunk_lengths = [len(c.page_content) for c in chunks_bonus]

print("📊 Chunk Size Distribution:")
print(f"   Smallest: {min(chunk_lengths)} chars")
print(f"   Largest: {max(chunk_lengths)} chars")
print(f"   Average: {sum(chunk_lengths)/len(chunk_lengths):.0f} chars")
print(f"   Median: {sorted(chunk_lengths)[len(chunk_lengths)//2]} chars")

# Count by size ranges
small = len([c for c in chunk_lengths if c < 500])
medium = len([c for c in chunk_lengths if 500 <= c < 900])
large = len([c for c in chunk_lengths if c >= 900])

print(f"\n📈 Size Distribution:")
print(f"   Small (<500 chars): {small} chunks ({small/len(chunk_lengths)*100:.1f}%)")
print(f"   Medium (500-900): {medium} chunks ({medium/len(chunk_lengths)*100:.1f}%)")
print(f"   Large (900-1000): {large} chunks ({large/len(chunk_lengths)*100:.1f}%)")


📊 Chunk Size Distribution:
   Smallest: 159 chars
   Largest: 997 chars
   Average: 811 chars
   Median: 943 chars

📈 Size Distribution:
   Small (<500 chars): 7 chunks (12.3%)
   Medium (500-900): 16 chunks (28.1%)
   Large (900-1000): 34 chunks (59.6%)


## Part 3: Embeddings

In [36]:
api_key = os.getenv("OPENAI_API_KEY")
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [37]:
# Storing embedding in Chroma vector database
db_bonus = Chroma.from_documents(chunks_bonus, embeddings, persist_directory="./chroma_db_BONUS")
print("ChromaDB created with document embeddings.")

ChromaDB created with document embeddings.


## Part 4: Retrieving and Preaparing document for Gen AI

In [38]:
user_question2 = "What are the bioanalytical validation requirements for measuring drug metabolites in plasma samples for IND submission?" # User question
retrieved_docs2 = db_bonus.similarity_search(user_question2, k=10) # k is the number of documents to retrieve

In [39]:
# Display top results
for i, doc in enumerate(retrieved_docs2[:3]): # Display top 3 results
    print(f"Document {i+1}:\n{doc.page_content}") # Display content
    print("-" * 80)  # Separator line

Document 1:
WuXi AppTec Laboratory Testing Division IND, ENABLED.
14
BIOANALYTICAL STUDIES
Bioanalytical studies are essential for IND submissions, as they provide quantitative measurements 
of drug concentrations and their metabolites in biological samples.  
Core bioanalytical studies typically include:
Method Development & Validation
Development and validation of analytical 
methods to quantify the investigational drug 
and its metabolites in biological samples (e.g., 
blood, plasma, urine). Validation ensures that the 
method is accurate, precise, and reproducible.
Determination of Bioavailability
Assessment of the fraction of the administered 
oral dose that reaches the systemic circulation  
and is available for therapeutic action.
Drug-Drug Interaction (DDI) Studies
Investigation of potential interactions 
between the drug and other drugs that may 
be co-administered.
Assessment of Stability
Evaluation of the stability and compatibility 
of the drug in biological samples during


In [43]:
def _get_document_prompt(docs):
    prompt = "\n"
    for doc in docs:
        prompt += "\nContent:\n"
        prompt += doc.page_content + "\n\n"
    return prompt

In [44]:
# Generate a formatted context from the retrieved documents
context = _get_document_prompt(retrieved_docs2)
print("Context formatted for GPT model.")

Context formatted for GPT model.


## Part 5: ChatBot Architecture

In [42]:
# Extract page numbers from retrieved documents
page_numbers2 = []
for doc in retrieved_docs2:
    page_num = doc.metadata.get('page', 'N/A')  # Get page number from metadata
    if page_num not in page_numbers2:  # Avoid duplicates
        page_numbers2.append(page_num)

# Convert to string (e.g., "15, 23, 31")
book_pages = ", ".join([str(p) for p in page_numbers2])

print(f"   Pages: {book_pages}")

   Pages: 13, 5, 16, 10, 11, 8, 4


In [45]:
# Construct prompt
prompt_bonus = f"""
You are a regulatory affairs and preclinical development specialist providing expert guidance on IND-enabling studies. Your role is to deliver precise, scientifically rigorous responses based on the WuXi AppTec IND Guide for pharmaceutical professionals engaged in drug development.

RETRIEVED CONTEXT:
{context}

SOURCE REFERENCE: Pages {book_pages}

EXPERT QUERY: 
{user_question2}

RESPONSE GUIDELINES:

1. SOURCE ATTRIBUTION
   - Preface your response with: "Based on the WuXi AppTec IND Guide (Pages {book_pages}):"
   - Cite specific page numbers for each key point using the format [Page X]

2. CONTENT REQUIREMENTS
   - Provide responses strictly within the scope of the retrieved context
   - Maintain scientific precision and use appropriate technical terminology
   - If the context provides partial information, clearly delineate:
     * What IS addressed in the source material
     * What information gaps remain
   - If the query falls outside the source scope, state: "This topic is not covered in the retrieved sections. Consult additional regulatory guidance or the complete WuXi AppTec IND Guide."

3. RESPONSE STRUCTURE
   For each major concept, use the following format:

   ### [Concept Name] [Page X]
   
   **Regulatory/Scientific Context:**  
   [Provide the technical definition and regulatory framework]
   
   **Implementation Requirements:**  
   [Detail specific study design elements, acceptance criteria, or methodological requirements]
   
   **Practical Considerations:**  
   [Industry best practices, common challenges, or strategic recommendations if mentioned in source]
   
   **Cross-References:**  
   [Note related topics, prerequisites, or downstream implications mentioned in the context]

4. EXPERT-LEVEL SPECIFICATIONS
   - Assume familiarity with: FDA regulations, ICH guidelines, GLP/GCP standards, ADME principles
   - Use appropriate abbreviations (IND, CRO, DMPK, PK/PD, AUC, Cmax, etc.) without extensive definition
   - Highlight regulatory implications and compliance considerations
   - Note any study dependencies or sequential requirements
   - Identify critical path activities when relevant

5. RESPONSE DEPTH
   - Provide comprehensive coverage (3-7 key points typical for complex queries)
   - Prioritize actionable, study-design-relevant information
   - Include quantitative parameters, timelines, or specifications when provided in source

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

EXPERT RESPONSE:

"""


## Part 6: Response

In [46]:
from IPython.display import Markdown, display, HTML
import datetime

# Generate AI response
client = openai.OpenAI()
model_params = {
    'model': 'gpt-4o-mini',
    'temperature': 0.3,
    'max_tokens': 1200,
    'top_p': 0.9,
    'frequency_penalty': 0.3,
    'presence_penalty': 0.1
}

messages = [{'role': 'user', 'content': prompt_bonus}]
completion = client.chat.completions.create(messages=messages, **model_params, timeout=120)
response = completion.choices[0].message.content

In [47]:
# Display professional response
# Print header with metadata
print("="*80)
print("🔬 PRECLINICAL IND DEVELOPMENT - EXPERT CONSULTATION")
print("="*80)
print()
print(f"📋 Query:")
print(f"   {user_question2}")
print()
print(f"📚 Source: WuXi AppTec IND Guide")
print(f"📄 Pages: {book_pages}")
print(f"🤖 Model: GPT-4o-mini (temp=0.3)")
print(f"⏱️  Generated: {datetime.datetime.now().strftime('%B %d, %Y at %H:%M IST')}")
print()
print("="*80)
print()

# Display the AI response as formatted markdown
display(Markdown(response))

# Footer
print()
print("-"*80)
print("⚠️  AI-generated response. Verify critical information with source material.")
print("-"*80)

🔬 PRECLINICAL IND DEVELOPMENT - EXPERT CONSULTATION

📋 Query:
   What are the bioanalytical validation requirements for measuring drug metabolites in plasma samples for IND submission?

📚 Source: WuXi AppTec IND Guide
📄 Pages: 13, 5, 16, 10, 11, 8, 4
🤖 Model: GPT-4o-mini (temp=0.3)
⏱️  Generated: January 24, 2026 at 19:19 IST




Based on the WuXi AppTec IND Guide (Pages 13, 5, 16, 10, 11, 8, 4):

### Bioanalytical Validation Requirements for Measuring Drug Metabolites in Plasma Samples [Page 13]

**Regulatory/Scientific Context:**  
Bioanalytical studies are critical for IND submissions as they provide quantitative measurements of drug concentrations and their metabolites in biological samples. The FDA has released guidance for the industry on bioanalytical method validation to ensure the quality of bioanalytical data generated during drug development [Page 5].

**Implementation Requirements:**  
1. **Method Development & Validation:**  
   - Analytical methods must be developed and validated to quantify both the investigational drug and its metabolites in plasma samples. Validation ensures accuracy, precision, specificity, sensitivity, and reproducibility of the assay [Page 13].
   
2. **Assay Characteristics:**  
   - The validation process should include assessments of linearity, limit of detection (LOD), limit of quantification (LOQ), and matrix effects. These characteristics are essential to confirm that the assay can reliably measure drug concentrations over the expected range [Page 16].

3. **Sample Stability Assessment:**  
   - Stability studies must evaluate how well the drug and its metabolites remain unchanged during sample collection, processing, and storage conditions. This ensures that results reflect true concentrations rather than artifacts from degradation [Page 10].

4. **Quality Control Samples:**  
   - Inclusion of quality control (QC) samples at multiple concentration levels throughout each analytical run is necessary to monitor assay performance and ensure reliability of results [Page 11].

5. **Data Analysis Procedures:**  
   - Statistical methods should be applied to analyze the data obtained from bioanalytical assays. This includes calculating pharmacokinetic parameters such as AUC (Area Under Curve) and Cmax (maximum concentration) based on validated methods [Page 8].

**Practical Considerations:**  
- It is crucial to adhere to Good Laboratory Practice (GLP) standards during bioanalytical method development and validation to ensure compliance with regulatory expectations.
- Common challenges include matrix effects that can interfere with accurate quantification; thus, careful selection of internal standards is recommended to mitigate these effects.

**Cross-References:**  
- Related topics include Drug Metabolism and Pharmacokinetics (DMPK) studies which provide insights into ADME properties that inform bioanalytical method design [Page 12]. Additionally, understanding potential drug-drug interactions (DDI) is essential when assessing metabolite profiles in clinical settings [Page 16]. 

In summary, rigorous validation of bioanalytical methods for measuring drug metabolites in plasma is paramount for supporting IND submissions and ensuring reliable pharmacokinetic data throughout clinical development.


--------------------------------------------------------------------------------
⚠️  AI-generated response. Verify critical information with source material.
--------------------------------------------------------------------------------
